# MSWEP: Weighted Precipitation Time Series

In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from exactextract import exact_extract
from shapely.geometry import box


In [ ]:
# Suppress noisy xattr errors from MSWEP files
sys.stderr = open(os.devnull, "w")


getfattr: /inputs/MSWEP_V280/Past/Daily/1989001.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989001.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989003.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989002.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989004.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989005.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989006.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989007.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989008.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989009.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989010.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989011.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989012.nc: Operation not supported
getfattr: /i

In [ ]:
# ── Inputs ──────────────────────────────────────────────────────────────────
shapefile_path   = Path("./caravan_shapefiles/camels/camels_basin_shapes.shp")
basins_txt_file  = Path("./basins_subset_test.txt")
mswep_dir        = Path("/inputs/MSWEP_V280/Past/Daily")

# Year range to process (adjust as needed)
YEAR_START, YEAR_END = 1989, 2019 #1989, 2008 

# ── Outputs ─────────────────────────────────────────────────────────────────
output_dir = Path("./mswep_precip_timeseries")
output_dir.mkdir(exist_ok=True)


In [4]:
gdf = gpd.read_file(shapefile_path)

with open(basins_txt_file) as f:
    selected_basins = [line.strip() for line in f.readlines()]

basins_gdf = gdf[gdf["gauge_id"].isin(selected_basins)].to_crs("EPSG:4326")
print(f"Loaded {len(basins_gdf)} basins")
basins_gdf.head()


Loaded 127 basins


,gauge_id,geometry
31,camels_01411300,"POLYGON ((-74.82836 39.38169, -74.83051 39.379..."
42,camels_01466500,"POLYGON ((-74.49563 39.89106, -74.4926 39.8866..."
46,camels_01487000,"MULTIPOLYGON (((-75.51965 38.87137, -75.51541 ..."
80,camels_01638480,"POLYGON ((-77.59039 39.2801, -77.58784 39.2789..."
82,camels_01644000,"POLYGON ((-77.79457 39.16487, -77.78732 39.159..."


In [5]:
# extracted_mswep = {
#     f.replace("_precip.csv", "")
#     for f in os.listdir("./precip_timeseries_mswep_weighted")
#     if f.endswith("_precip.csv")
# }

# basins_gdf_filtered = basins_gdf[~basins_gdf["gauge_id"].isin(extracted_mswep)]

# print(f"Original: {len(basins_gdf)}, Extracted: {len(extracted_mswep)}, Remaining: {len(basins_gdf_filtered)}")

In [6]:
# basins_gdf = basins_gdf_filtered

In [7]:
gauges = list(basins_gdf["gauge_id"])

gdf_all = gpd.GeoDataFrame(
    {"gauge_id": gauges},
    geometry=list(basins_gdf.geometry),
    crs="EPSG:4326",
)
gdf_all.head()


,gauge_id,geometry
0,camels_01411300,"POLYGON ((-74.82836 39.38169, -74.83051 39.379..."
1,camels_01466500,"POLYGON ((-74.49563 39.89106, -74.4926 39.8866..."
2,camels_01487000,"MULTIPOLYGON (((-75.51965 38.87137, -75.51541 ..."
3,camels_01638480,"POLYGON ((-77.59039 39.2801, -77.58784 39.2789..."
4,camels_01644000,"POLYGON ((-77.79457 39.16487, -77.78732 39.159..."


In [8]:
all_files = sorted(mswep_dir.glob("*.nc"))

# Filter by year encoded in the filename (first 4 characters)
files = [
    f for f in all_files
    if YEAR_START <= int(f.stem[:4]) <= YEAR_END
]

print(f"Total MSWEP files to process: {len(files)}")
files[:3]


Total MSWEP files to process: 11322


[PosixPath('/inputs/MSWEP_V280/Past/Daily/1989001.nc'),
 PosixPath('/inputs/MSWEP_V280/Past/Daily/1989002.nc'),
 PosixPath('/inputs/MSWEP_V280/Past/Daily/1989003.nc')]

In [10]:
def process_file_weighted(file_path, gdf_all, gauges):
    stem = file_path.stem
    year = int(stem[:4])
    doy  = int(stem[4:])
    date = pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=doy - 1)

    ds = xr.open_dataset(file_path)
    da = ds["precipitation"].squeeze()

    rename_map = {}
    if "lon" in da.dims: rename_map["lon"] = "x"
    if "lat" in da.dims: rename_map["lat"] = "y"
    da = da.rename(rename_map)
    da = da.rio.write_crs("EPSG:4326")

    result = exact_extract(da, gdf_all, ["mean"], output="pandas")
    result["mean"] = result["mean"].where(
        (result["mean"] != -9999) & (result["mean"] >= 0), other=np.nan
    )
    ds.close()
    return date, dict(zip(gauges, result["mean"].values))


In [11]:
# Run ONE file synchronously to confirm the function itself works
date, values = process_file_weighted(files[0], gdf_all, gauges)
print(date, list(values.items())[:3])

1989-01-01 00:00:00 [('camels_01411300', np.float64(3.2148521470474924)), ('camels_01466500', np.float64(2.1081729654427166)), ('camels_01487000', np.float64(6.2382447854460565))]


In [13]:
all_results = []
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_file_weighted, f, gdf_all, gauges) for f in files]
    for i, future in enumerate(futures, 1):
        date, values = future.result()
        all_results.append((date, values))
        print(f"{date.date()} processed ({i}/{len(files)})")

1989-01-01 processed (1/11322)
1989-01-02 processed (2/11322)
1989-01-03 processed (3/11322)
1989-01-04 processed (4/11322)
1989-01-05 processed (5/11322)
1989-01-06 processed (6/11322)
1989-01-07 processed (7/11322)
1989-01-08 processed (8/11322)
1989-01-09 processed (9/11322)
1989-01-10 processed (10/11322)
1989-01-11 processed (11/11322)
1989-01-12 processed (12/11322)
1989-01-13 processed (13/11322)
1989-01-14 processed (14/11322)
1989-01-15 processed (15/11322)
1989-01-16 processed (16/11322)
1989-01-17 processed (17/11322)
1989-01-18 processed (18/11322)
1989-01-19 processed (19/11322)
1989-01-20 processed (20/11322)
1989-01-21 processed (21/11322)
1989-01-22 processed (22/11322)
1989-01-23 processed (23/11322)
1989-01-24 processed (24/11322)
1989-01-25 processed (25/11322)
1989-01-26 processed (26/11322)
1989-01-27 processed (27/11322)
1989-01-28 processed (28/11322)
1989-01-29 processed (29/11322)
1989-01-30 processed (30/11322)
1989-01-31 processed (31/11322)
1989-02-01 proces

In [14]:
# Collect per-basin lists
basin_series = {gauge: [] for gauge in gauges}
dates = []

for date, values in all_results:
    dates.append(date)
    for gauge, val in values.items():
        basin_series[gauge].append(val)

dates = pd.to_datetime(dates)

# Save one CSV per basin
for gauge, values in basin_series.items():
    ts = pd.Series(values, index=dates, name="precipitation").sort_index()
    out_path = output_dir / f"{gauge}_precip.csv"
    ts.to_csv(out_path)
    print(f"Saved: {out_path}")


Saved: precip_timeseries_mswep_weighted_2/camels_01411300_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01466500_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01487000_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01638480_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01644000_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01666500_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01667500_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01669000_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_01669520_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_02027000_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_02038850_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_02046000_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_02053200_precip.csv
Saved: precip_timeseries_mswep_weighted_2/camels_02053800_precip.csv
Saved: precip_timeseries_mswep_wei